# 00 — Orquestador Medallion

Ejecuta en orden los notebooks del flujo Bronze → Silver → Gold **dentro de una única sesión Spark**, de forma secuencial, con reintentos y un resumen final de estado y duración.

## Requisitos
- Lakehouse predeterminado: **`LH_Silver`** (el mismo que el resto de notebooks).
- Los notebooks hijos deben existir en el mismo workspace con los nombres indicados en la lista. El Lakehouse predeterminado de este orquestador debe ser `LH_Silver`.
- No uses `%%configure` en los notebooks hijos; si lo necesitas, ponlo solo en este orquestador.

## Funcionamiento
- Si un notebook falla, se reintenta el número de veces configurado.
- Si agota los reintentos, **se detiene la cadena** (los siguientes no se ejecutan) y el notebook termina en error, para que un pipeline o una programación lo marquen como fallido.
- Al final se muestra una tabla con el estado y la duración de cada paso.

## 1. Configuración

In [ ]:
# Notebooks a ejecutar, en orden. La validación Gold es el último paso obligatorio.
NOTEBOOKS = [
    "01_bronze_to_silver",
    "02_silver_to_gold",
    "03_validar_gold",
]

TIMEOUT_SEGUNDOS = 1800   # tiempo máximo por notebook
REINTENTOS = 2            # reintentos adicionales tras el primer fallo
ESPERA_SEGUNDOS = 60      # pausa entre reintentos

# Parámetros opcionales que se pasan a todos los notebooks hijos
# (solo se aplican si el hijo tiene una celda marcada como "Parameter cell").
PARAMETROS = {}

## 2. Función de ejecución con reintentos

In [ ]:
import time
from datetime import datetime


def ejecutar_notebook(nombre, timeout, reintentos, espera, parametros):
    """Ejecuta un notebook hijo y devuelve un dict con el resultado."""
    inicio = time.time()
    intentos_realizados = 0
    ultimo_error = None

    for intento in range(1, reintentos + 2):
        intentos_realizados = intento
        try:
            print(f"[{datetime.now():%H:%M:%S}] Ejecutando {nombre} (intento {intento}/{reintentos + 1})...")
            salida = notebookutils.notebook.run(nombre, timeout, parametros)
            return {
                "notebook": nombre,
                "estado": "OK",
                "intentos": intentos_realizados,
                "duracion_s": round(time.time() - inicio, 1),
                "detalle": str(salida)[:200],
            }
        except Exception as e:
            ultimo_error = str(e)
            print(f"[{datetime.now():%H:%M:%S}] {nombre} falló en el intento {intento}: {ultimo_error[:300]}")
            if intento <= reintentos:
                print(f"Reintentando en {espera} s...")
                time.sleep(espera)

    return {
        "notebook": nombre,
        "estado": "ERROR",
        "intentos": intentos_realizados,
        "duracion_s": round(time.time() - inicio, 1),
        "detalle": (ultimo_error or "")[:200],
    }

## 3. Ejecución de la cadena

Los notebooks se ejecutan de forma secuencial. Si uno falla definitivamente, se interrumpe la cadena.

In [ ]:
resultados = []
cadena_ok = True

for nb_nombre in NOTEBOOKS:
    if not cadena_ok:
        resultados.append({
            "notebook": nb_nombre,
            "estado": "OMITIDO",
            "intentos": 0,
            "duracion_s": 0.0,
            "detalle": "No se ejecutó porque un paso anterior falló.",
        })
        continue

    res = ejecutar_notebook(nb_nombre, TIMEOUT_SEGUNDOS, REINTENTOS, ESPERA_SEGUNDOS, PARAMETROS)
    resultados.append(res)
    if res["estado"] != "OK":
        cadena_ok = False

## 4. Resumen

In [ ]:
resumen_df = spark.createDataFrame(resultados)
display(resumen_df.select("notebook", "estado", "intentos", "duracion_s", "detalle"))

total = round(sum(r["duracion_s"] for r in resultados), 1)
print(f"Duración total: {total} s")

## 5. Resultado final

Si algún paso ha fallado, el orquestador lanza una excepción para que la ejecución quede marcada como **fallida** en el Monitoring hub o en el pipeline que lo invoque.

In [ ]:
fallidos = [r for r in resultados if r["estado"] == "ERROR"]
if fallidos:
    raise RuntimeError(
        "La cadena falló en: " + ", ".join(r["notebook"] for r in fallidos)
        + ". Revisa el resumen y los snapshots del notebook hijo."
    )

print("Cadena completada correctamente.")